# 01 - Ingestão, Reconciliação e Gold

Fluxo correto: Bronze (base antiga + base nova) -> Silver unificado -> Gold de modelagem.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
BASE_DIR = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.pipeline.schema_map import (
    load_and_normalize_workbook,
    load_and_normalize_old_base,
    reconcile_bases,
    validate_normalized_data,
)
from src.pipeline.relational_features import build_relational_features, enrich_with_relational_features
from src.features.risk_target import build_risk_dataset


In [ ]:
# Caminhos bronze
new_workbook = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
old_csv = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'Bases antigas' / 'PEDE_PASSOS_DATASET_FIAP.csv'

print('new exists?', new_workbook.exists(), new_workbook)
print('old exists?', old_csv.exists(), old_csv)

In [ ]:
# Bronze normalizado
df_new = load_and_normalize_workbook(str(new_workbook))
df_old = load_and_normalize_old_base(str(old_csv))

print('new rows:', len(df_new), '| anos:', sorted(df_new['ano_referencia'].unique()))
print('old rows:', len(df_old), '| anos:', sorted(df_old['ano_referencia'].unique()))

In [ ]:
# Silver reconciliado (2022 prioriza base nova)
df_silver = reconcile_bases(df_new, df_old)
df_valid, df_invalid = validate_normalized_data(df_silver)

print('silver rows:', len(df_silver))
print('valid rows:', len(df_valid), '| invalid rows:', len(df_invalid))
display(df_silver.groupby(['ano_referencia','source_system']).size().to_frame('linhas'))

In [ ]:
# Features relacionais (TbDiario, TbFaseNotaAluno, TbHistoricoNotas)
bases_antigas_dir = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'Bases antigas'
rel_base = next(p for p in bases_antigas_dir.iterdir() if p.is_dir() and p.name.lower().startswith('base de dados - passos'))
df_rel_feat = build_relational_features(rel_base)
print('rel_features rows:', len(df_rel_feat))
display(df_rel_feat.head())


In [ ]:
# Enriquecimento do silver com features relacionais
df_silver_enriched = enrich_with_relational_features(df_silver, df_rel_feat)
df_valid, df_invalid = validate_normalized_data(df_silver_enriched)
print('silver enriched rows:', len(df_silver_enriched))
print('valid rows:', len(df_valid), '| invalid rows:', len(df_invalid))
extra_cols = [c for c in df_silver_enriched.columns if c.startswith(('freq_','fase_','hist_'))]
print('cols relacionais:', extra_cols)


In [ ]:
# Diagnóstico rápido de nulos (silver validado)
display((df_valid.isna().mean() * 100).sort_values(ascending=False).to_frame('%_nulos').head(25))


In [ ]:
# Gold inicial de risco
risk_df = build_risk_dataset(df_valid).copy()
print('risk_df shape:', risk_df.shape)
print('taxa_risco:', round(risk_df['target_risco'].mean() * 100, 2), '%')

In [ ]:
# Features baseline
features_base = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'inde', 'defasagem', 'ano_referencia']
target_col = 'target_risco'

drop_cols = ['fase_ideal', 'atingiu_pv', 'escola', 'status_ativo', 'ingles']
for c in drop_cols:
    if c in risk_df.columns:
        risk_df.drop(columns=c, inplace=True)

for c in features_base:
    if c in risk_df.columns:
        risk_df[c] = risk_df[c].fillna(risk_df[c].median())

display((risk_df[features_base + [target_col]].isna().mean() * 100).to_frame('%_nulos'))
display(risk_df[features_base + [target_col]].head())

In [ ]:
# Persistência silver e gold
silver_dir = PROJECT_ROOT / 'data' / 'silver'
gold_dir = PROJECT_ROOT / 'data' / 'gold'
silver_dir.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)

silver_file = silver_dir / 'base_unificada_validada_enriquecida.csv'
invalid_file = silver_dir / 'base_unificada_invalidos.csv'
gold_file = gold_dir / 'base_modelagem_risco.csv'

df_valid.to_csv(silver_file, index=False)
df_invalid.to_csv(invalid_file, index=False)
risk_df.to_csv(gold_file, index=False)

print('silver:', silver_file)
print('invalid:', invalid_file)
print('gold:', gold_file)

